# Spotify Wrapped pero Real
## Lo que el Wrapped oficial no te cuenta sobre cómo escuchas música

---

**Autora:** aroaxinping  
**Fecha:** Abril 2026  
**Herramientas:** Python · pandas · scikit-learn · matplotlib · seaborn

---

### Resumen

El Wrapped de Spotify es bonito, pero superficial. Te dice tu top 5 artistas y poco más. Este análisis usa el historial extendido de streaming (que puedes solicitar en la configuración de privacidad de Spotify) para responder preguntas que el Wrapped nunca toca: ¿cómo son tus sesiones de escucha? ¿cuánto decide el algoritmo por ti? ¿se pueden detectar "épocas" musicales en tu historial?

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Estilo
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#333',
    'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#999',
    'ytick.color': '#999',
    'grid.color': '#333',
    'grid.alpha': 0.3,
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})

# Paleta
C_NARANJA = '#e85d04'
C_AZUL = '#6a9ad4'
C_FONDO = '#0f0f0f'
C_GRIS = '#888888'
C_VERDE = '#4ecdc4'
C_ROJO = '#ff6b6b'

print('Entorno configurado.')

---
## 1. Datos

Cargamos el historial procesado. Si no existe (porque no has descargado tus datos de Spotify), generamos un dataset sintético realista con ~5000 eventos de escucha en 6 meses.

In [ ]:
# -----------------------------------------------------------------------
# Intentar cargar datos reales procesados
# Si no existen, generar dataset sintético realista
# -----------------------------------------------------------------------

import sys, os
sys.path.insert(0, os.path.abspath('../src'))

DATA_PATH = Path('../data/processed/spotify_history.csv')

def load_or_generate():
    if DATA_PATH.exists():
        df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
        print(f'[OK] Datos cargados: {len(df)} eventos')
        print(f'     {df["artist"].nunique()} artistas, {df["track"].nunique()} canciones')
        print(f'     Periodo: {df["timestamp"].min().date()} — {df["timestamp"].max().date()}')
        return df

    print('[INFO] No se encontró spotify_history.csv — generando datos sintéticos')
    from fetch_spotify_data import generate_synthetic_data
    df = generate_synthetic_data(n_events=5000, n_artists=50, n_tracks=200)
    print(f'[OK] Dataset sintético: {len(df)} eventos, {df["artist"].nunique()} artistas')
    return df

df = load_or_generate()

# Columnas derivadas
df['hours_played'] = df['ms_played'] / 3_600_000
df['min_played'] = df['ms_played'] / 60_000
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['dow'] = df['timestamp'].dt.dayofweek  # 0=lunes
df['dow_name'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.to_period('M')
df['week'] = df['timestamp'].dt.isocalendar().week.astype(int)

print(f'\nResumen:')
print(f'  Total horas escuchadas: {df["hours_played"].sum():.1f} h')
print(f'  Media diaria: {df.groupby("date")["hours_played"].sum().mean():.1f} h/día')

---
## 2. Patrones temporales

> **Pregunta:** ¿A qué hora del día y qué día de la semana escucho más música?

In [ ]:
# Heatmap: hora × día de la semana
dow_labels = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
pivot = df.groupby(['dow', 'hour'])['hours_played'].sum().reset_index()
heatmap_data = pivot.pivot(index='dow', columns='hour', values='hours_played').fillna(0)
heatmap_data.index = [dow_labels[i] for i in heatmap_data.index]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    heatmap_data, cmap='YlOrRd', linewidths=0.5, linecolor='#333',
    cbar_kws={'label': 'Horas escuchadas'},
    ax=ax
)
ax.set_title('¿Cuándo escucho música? (horas totales por franja)', color='white')
ax.set_xlabel('Hora del día')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Listening time por mes
monthly = df.groupby('month')['hours_played'].sum().reset_index()
monthly['month_str'] = monthly['month'].astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(monthly['month_str'], monthly['hours_played'], color=C_NARANJA, edgecolor='#333')
for bar, val in zip(bars, monthly['hours_played']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.0f}h', ha='center', va='bottom', color='white', fontsize=11)
ax.set_title('Horas de escucha por mes')
ax.set_xlabel('Mes')
ax.set_ylabel('Horas')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Top artistas y canciones

> **Pregunta:** ¿Cuáles son mis artistas y canciones más escuchados de verdad (por tiempo, no por plays)?

In [ ]:
# Top 15 artistas por tiempo de escucha
top_artists = df.groupby('artist')['hours_played'].sum().nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_artists.index, top_artists.values, color=C_NARANJA, edgecolor='#333')
for bar, val in zip(bars, top_artists.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}h', va='center', color='white', fontsize=10)
ax.set_title('Top 15 artistas por horas escuchadas')
ax.set_xlabel('Horas')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 canciones por tiempo de escucha
top_tracks = df.groupby(['track', 'artist']).agg(
    hours=('hours_played', 'sum'),
    plays=('track', 'count')
).nlargest(15, 'hours').sort_values('hours')

labels = [f'{t[0]}\n({t[1]})' for t in top_tracks.index]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(labels, top_tracks['hours'].values, color=C_AZUL, edgecolor='#333')
for bar, val, plays in zip(bars, top_tracks['hours'].values, top_tracks['plays'].values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}h ({plays} plays)', va='center', color='white', fontsize=9)
ax.set_title('Top 15 canciones por horas escuchadas')
ax.set_xlabel('Horas')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Skip rate por artista (top 20 más escuchados)
top20_artists = df.groupby('artist')['hours_played'].sum().nlargest(20).index
artist_stats = df[df['artist'].isin(top20_artists)].groupby('artist').agg(
    skip_rate=('skipped', 'mean'),
    total_plays=('track', 'count'),
    hours=('hours_played', 'sum')
).sort_values('skip_rate')

fig, ax = plt.subplots(figsize=(12, 7))
colors = [C_ROJO if r > 0.2 else C_VERDE for r in artist_stats['skip_rate']]
bars = ax.barh(artist_stats.index, artist_stats['skip_rate'] * 100, color=colors, edgecolor='#333')
for bar, val in zip(bars, artist_stats['skip_rate'] * 100):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', color='white', fontsize=9)
ax.set_title('Skip rate por artista (top 20 más escuchados)')
ax.set_xlabel('% de canciones skipeadas')
ax.axvline(x=20, color=C_GRIS, linestyle='--', alpha=0.5, label='umbral 20%')
ax.legend(facecolor='#1a1a1a', edgecolor='#333')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Sesiones de escucha

> **Pregunta:** ¿Cómo son mis sesiones? ¿Largas y enfocadas o cortas y dispersas?

Definimos una sesión como un bloque continuo de escucha. Si hay un gap de más de 30 minutos entre dos canciones consecutivas, es una nueva sesión.

In [ ]:
# Detectar sesiones (gap > 30 min = nueva sesión)
df_sorted = df.sort_values('timestamp').reset_index(drop=True)
gaps = df_sorted['timestamp'].diff()
session_break = gaps > pd.Timedelta(minutes=30)
df_sorted['session_id'] = session_break.cumsum()

sessions = df_sorted.groupby('session_id').agg(
    start=('timestamp', 'min'),
    end=('timestamp', 'max'),
    n_tracks=('track', 'count'),
    n_artists=('artist', 'nunique'),
    total_min=('min_played', 'sum'),
    total_hours=('hours_played', 'sum'),
).reset_index()
sessions['duration_min'] = (sessions['end'] - sessions['start']).dt.total_seconds() / 60

print(f'Total sesiones detectadas: {len(sessions)}')
print(f'Duración media: {sessions["duration_min"].mean():.0f} min')
print(f'Canciones por sesión (media): {sessions["n_tracks"].mean():.1f}')
print(f'Sesión más larga: {sessions["duration_min"].max():.0f} min ({sessions["n_tracks"].max()} canciones)')

In [ ]:
# Distribución de duración de sesiones y tracks por sesión
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Duración
ax1.hist(sessions['duration_min'].clip(upper=180), bins=40, color=C_NARANJA, edgecolor='#333', alpha=0.9)
ax1.axvline(sessions['duration_min'].median(), color=C_AZUL, linestyle='--', linewidth=2,
            label=f'Mediana: {sessions["duration_min"].median():.0f} min')
ax1.set_title('Duración de las sesiones')
ax1.set_xlabel('Minutos')
ax1.set_ylabel('Frecuencia')
ax1.legend(facecolor='#1a1a1a', edgecolor='#333')
ax1.grid(axis='y', alpha=0.3)

# Tracks por sesión
ax2.hist(sessions['n_tracks'].clip(upper=50), bins=30, color=C_AZUL, edgecolor='#333', alpha=0.9)
ax2.axvline(sessions['n_tracks'].median(), color=C_NARANJA, linestyle='--', linewidth=2,
            label=f'Mediana: {sessions["n_tracks"].median():.0f} tracks')
ax2.set_title('Canciones por sesión')
ax2.set_xlabel('Número de canciones')
ax2.set_ylabel('Frecuencia')
ax2.legend(facecolor='#1a1a1a', edgecolor='#333')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Clustering de épocas musicales

> **Pregunta:** ¿Se pueden detectar "épocas" en mi historial de escucha?

Agrupamos el historial por mes y creamos un perfil de escucha mensual (% de tiempo por artista). Luego aplicamos K-means para detectar clusters — cada cluster es una "era" musical.

In [ ]:
# Perfil mensual: % de tiempo por artista (top 20 para reducir dimensionalidad)
top20 = df.groupby('artist')['hours_played'].sum().nlargest(20).index
df_top = df[df['artist'].isin(top20)].copy()

monthly_profile = df_top.groupby(['month', 'artist'])['hours_played'].sum().reset_index()
monthly_pivot = monthly_profile.pivot_table(
    index='month', columns='artist', values='hours_played', fill_value=0
)
# Normalizar a % por fila
monthly_pct = monthly_pivot.div(monthly_pivot.sum(axis=1), axis=0)

# K-means clustering
n_clusters = min(3, len(monthly_pct))  # máximo 3 eras o menos si pocos meses
scaler = StandardScaler()
X = scaler.fit_transform(monthly_pct.values)

km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
monthly_pct['era'] = km.fit_predict(X)

# Top artista por era
print('Épocas musicales detectadas:\n')
for era in range(n_clusters):
    months = monthly_pct[monthly_pct['era'] == era].index.tolist()
    era_data = monthly_pivot.loc[monthly_pct['era'] == era]
    top3 = era_data.sum().nlargest(3)
    month_range = f'{months[0]} — {months[-1]}' if len(months) > 1 else str(months[0])
    print(f'  Era {era + 1} ({month_range}):')
    for artist, hours in top3.items():
        print(f'    - {artist}: {hours:.1f}h')
    print()

In [ ]:
# Timeline de eras musicales
era_colors = [C_NARANJA, C_AZUL, C_VERDE, C_ROJO]
month_labels = [str(m) for m in monthly_pct.index]

fig, ax = plt.subplots(figsize=(14, 4))
for i, (month, row) in enumerate(monthly_pct.iterrows()):
    era = int(row['era'])
    ax.barh(0, 1, left=i, color=era_colors[era % len(era_colors)],
            edgecolor='#333', linewidth=0.5, height=0.6)
    ax.text(i + 0.5, 0, str(month), ha='center', va='center', color='white',
            fontsize=9, fontweight='bold')

# Leyenda
patches = [mpatches.Patch(color=era_colors[i], label=f'Era {i+1}') for i in range(n_clusters)]
ax.legend(handles=patches, loc='upper right', facecolor='#1a1a1a', edgecolor='#333')
ax.set_title('Timeline de épocas musicales (clustering K-means)')
ax.set_yticks([])
ax.set_xlim(-0.2, len(month_labels) + 0.2)
ax.set_xlabel('Mes')
plt.tight_layout()
plt.show()

# Heatmap de perfil mensual
fig, ax = plt.subplots(figsize=(16, 6))
plot_data = monthly_pct.drop(columns='era')
# Solo mostrar top 10 artistas para legibilidad
top10_cols = plot_data.sum().nlargest(10).index
sns.heatmap(plot_data[top10_cols].T, cmap='YlOrRd', linewidths=0.5, linecolor='#333',
            xticklabels=[str(m) for m in plot_data.index],
            cbar_kws={'label': '% del tiempo'}, ax=ax)
ax.set_title('Perfil de escucha mensual (top 10 artistas)')
ax.set_ylabel('')
ax.set_xlabel('Mes')
plt.tight_layout()
plt.show()

---
## 6. Shuffle y skips

> **Pregunta:** ¿Cuánto controlo yo lo que escucho vs cuánto decide el algoritmo?

In [ ]:
# Shuffle vs selección manual
shuffle_pct = df['shuffle'].mean() * 100
manual_pct = 100 - shuffle_pct

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart shuffle
sizes = [shuffle_pct, manual_pct]
labels = [f'Shuffle\n{shuffle_pct:.0f}%', f'Manual\n{manual_pct:.0f}%']
colors_pie = [C_AZUL, C_NARANJA]
wedges, texts = ax1.pie(sizes, labels=labels, colors=colors_pie,
                         startangle=90, textprops={'color': 'white', 'fontsize': 13})
ax1.set_title('Shuffle vs selección manual')

# Skip rate general + por contexto
skip_total = df['skipped'].mean() * 100
skip_shuffle = df[df['shuffle'] == True]['skipped'].mean() * 100
skip_manual = df[df['shuffle'] == False]['skipped'].mean() * 100

bars_data = {'Total': skip_total, 'En shuffle': skip_shuffle, 'Selección manual': skip_manual}
bar_colors = [C_GRIS, C_AZUL, C_NARANJA]
bars = ax2.bar(bars_data.keys(), bars_data.values(), color=bar_colors, edgecolor='#333')
for bar, val in zip(bars, bars_data.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', color='white', fontsize=12)
ax2.set_title('Skip rate: shuffle vs manual')
ax2.set_ylabel('% skipeado')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Escucha en shuffle: {shuffle_pct:.1f}%')
print(f'Skip rate total: {skip_total:.1f}%')
print(f'Skip rate en shuffle: {skip_shuffle:.1f}% | Manual: {skip_manual:.1f}%')

In [ ]:
# Canciones que siempre skipeo vs nunca skipeo
track_skip = df.groupby(['track', 'artist']).agg(
    plays=('track', 'count'),
    skips=('skipped', 'sum'),
    skip_rate=('skipped', 'mean'),
    hours=('hours_played', 'sum'),
).reset_index()

# Solo canciones con >= 5 plays para que sea significativo
track_skip = track_skip[track_skip['plays'] >= 5]

# Siempre skipeo (skip rate > 50%, mínimo 5 plays)
always_skip = track_skip[track_skip['skip_rate'] > 0.5].nlargest(10, 'plays')
# Nunca skipeo (skip rate = 0, más plays)
never_skip = track_skip[track_skip['skip_rate'] == 0].nlargest(10, 'plays')

print('Canciones que casi siempre skipeo (>50% skip rate, min 5 plays):')
if len(always_skip) > 0:
    for _, row in always_skip.iterrows():
        print(f'  {row["track"]} — {row["artist"]} ({row["plays"]} plays, {row["skip_rate"]*100:.0f}% skip)')
else:
    print('  Ninguna con suficientes plays')

print(f'\nCanciones que NUNCA skipeo (0% skip rate, min 5 plays):')
if len(never_skip) > 0:
    for _, row in never_skip.iterrows():
        print(f'  {row["track"]} — {row["artist"]} ({row["plays"]} plays, {row["hours"]:.1f}h)')
else:
    print('  Ninguna con suficientes plays')

---
## 7. Sintesis y conclusiones

In [ ]:
# Tabla resumen
total_hours = df['hours_played'].sum()
total_tracks = len(df)
unique_artists = df['artist'].nunique()
unique_tracks = df['track'].nunique()
top_artist = df.groupby('artist')['hours_played'].sum().idxmax()
top_track = df.groupby('track')['hours_played'].sum().idxmax()
peak_hour = df.groupby('hour')['hours_played'].sum().idxmax()
avg_session_min = sessions['duration_min'].mean()

summary = pd.DataFrame({
    'Métrica': [
        'Total horas escuchadas',
        'Total reproducciones',
        'Artistas únicos',
        'Canciones únicas',
        'Artista #1 (por tiempo)',
        'Canción #1 (por tiempo)',
        'Hora pico de escucha',
        'Sesión media',
        '% shuffle',
        '% skip rate',
        'Épocas detectadas (K-means)',
    ],
    'Valor': [
        f'{total_hours:.0f} h',
        f'{total_tracks:,}',
        unique_artists,
        unique_tracks,
        top_artist,
        top_track,
        f'{peak_hour}:00',
        f'{avg_session_min:.0f} min',
        f'{shuffle_pct:.0f}%',
        f'{skip_total:.1f}%',
        n_clusters,
    ]
})

print(summary.to_markdown(index=False))

---

**Esto es lo que Spotify sabe de ti pero no te cuenta.** El Wrapped oficial es marketing — cinco artistas bonitos y un color. Con los datos extendidos puedes ver patrones reales: tus sesiones, tus épocas, cuánto decides tú y cuánto decide el algoritmo.

Para descargar tus propios datos: [Spotify Privacy Settings](https://www.spotify.com/account/privacy/) → "Extended streaming history". Tarda unos días en llegar.